# Introductory Statistics – Part 2: Probability & Distributions

Welcome to Part 2. Building on the descriptive statistics from Part 1, we now turn to **probability** — the mathematical foundation that lets us move from describing data to making inferences.

**By the end of this notebook you will be able to:**

- Use Python/SciPy as an alternative to point-and-click tools like Minitab
- Distinguish discrete and continuous probability distributions
- Compute probabilities and visualise the **binomial distribution**
- Compute probabilities and visualise the **normal distribution**

**Topics covered:**

5. Using a statistical package (Minitab overview + Python equivalents)  
6. Probability and probability distributions  
7. Binomial distribution  
8. Normal distribution

> **Prerequisite:** Part 1 – Descriptive Statistics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binom, norm

sns.set(style="whitegrid")
np.random.seed(42)

---
## 5. Using a Statistical Package — Minitab and Python

**Minitab** is a point-and-click statistical software package widely used in industry and academia. It provides menus for:

| Task | Minitab menu | Python equivalent |
|---|---|---|
| Descriptive statistics | Stat → Basic Statistics | `pandas`, `numpy` |
| Probability distributions | Calc → Probability Distributions | `scipy.stats` |
| Hypothesis tests | Stat → Basic Statistics | `scipy.stats` |
| Regression & ANOVA | Stat → Regression / ANOVA | `statsmodels` |

Throughout this series we use Python, which gives you the same results with full transparency and reproducibility.

The cell below mirrors what Minitab's **Calc → Probability Distributions → Binomial** dialog would compute.

In [ ]:
# Suppose X ~ Binomial(n=10, p=0.3)

n, p = 10, 0.3

prob_exactly_3 = binom.pmf(3, n, p)   # P(X = 3)
prob_at_most_3 = binom.cdf(3, n, p)   # P(X ≤ 3)
prob_more_than_3 = 1 - prob_at_most_3 # P(X > 3)

print(f"X ~ Binomial(n={n}, p={p})")
print(f"  P(X = 3)  = {prob_exactly_3:.4f}")
print(f"  P(X ≤ 3)  = {prob_at_most_3:.4f}")
print(f"  P(X > 3)  = {prob_more_than_3:.4f}")

---
## 6. Probability and Probability Distributions

**Probability** is a number between 0 and 1 that quantifies how likely an event is.

A **random variable** $X$ assigns a numerical value to each outcome of a random experiment. Its **probability distribution** specifies all possible values and their associated probabilities.

### Two major types

| Type | Description | Example |
|---|---|---|
| **Discrete** | Countable outcomes; described by a PMF $P(X = x)$ | Number of heads in 10 flips |
| **Continuous** | Uncountably many outcomes; described by a PDF $f(x)$ | Height of a randomly chosen person |

For a discrete distribution, probabilities sum to 1: $\sum_x P(X = x) = 1$.

For a continuous distribution, the area under the density curve equals 1: $\int_{-\infty}^{\infty} f(x)\, dx = 1$.

In [ ]:
# Discrete example: rolling a fair six-sided die
# Each outcome has equal probability 1/6.

outcomes = np.arange(1, 7)
probs = np.repeat(1/6, 6)

plt.figure(figsize=(6, 4))
plt.bar(outcomes, probs, color="skyblue", edgecolor="white")
plt.axhline(1/6, color="red", linestyle="--", alpha=0.6, label="P = 1/6")
plt.title("Probability Distribution of a Fair Die")
plt.xlabel("Outcome")
plt.ylabel("Probability")
plt.ylim(0, 0.25)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Sum of probabilities: {probs.sum():.1f}  (must equal 1)")

In [ ]:
# Continuous example: standard normal distribution N(0, 1)
# The curve shows density — probabilities correspond to areas under it.

x = np.linspace(-4, 4, 400)
y = norm.pdf(x, loc=0, scale=1)

plt.figure(figsize=(6, 4))
plt.plot(x, y, color="darkred", linewidth=2)
plt.fill_between(x, y, where=(x >= -1) & (x <= 1), alpha=0.25, color="darkred",
                 label=f"P(-1 ≤ X ≤ 1) = {norm.cdf(1) - norm.cdf(-1):.3f}")
plt.title("Standard Normal Distribution $N(0, 1)$")
plt.xlabel("$x$")
plt.ylabel("Density $f(x)$")
plt.legend()
plt.tight_layout()
plt.show()

---
## 7. Binomial Distribution

The **binomial distribution** $X \sim \text{Binomial}(n, p)$ counts the number of successes in $n$ independent trials where each trial has probability $p$ of success.

### Conditions (BINS)
1. **B**inary outcomes — each trial is success or failure
2. **I**ndependent trials
3. **N**umber of trials $n$ is fixed in advance
4. **S**ame probability $p$ for each trial

### Key formulas

| Quantity | Formula |
|---|---|
| PMF | $P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$ |
| Mean | $\mu = np$ |
| Variance | $\sigma^2 = np(1-p)$ |
| Std dev | $\sigma = \sqrt{np(1-p)}$ |

**Example:** number of heads in 10 fair coin flips → $X \sim \text{Binomial}(10,\, 0.5)$.

In [ ]:
# Binomial PMF for n=10, p=0.5
n, p = 10, 0.5
x = np.arange(0, n + 1)
pmf_vals = binom.pmf(x, n, p)

mu_binom  = n * p
std_binom = np.sqrt(n * p * (1 - p))

plt.figure(figsize=(7, 4))
plt.stem(x, pmf_vals, basefmt=" ")
plt.axvline(mu_binom, color="red", linestyle="--", label=f"Mean $\\mu = {mu_binom}$")
plt.title(f"Binomial Distribution: $n={n},\\, p={p}$")
plt.xlabel("Number of Successes $k$")
plt.ylabel("$P(X = k)$")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean (μ = np): {mu_binom}")
print(f"Std dev (σ = √(np(1−p))): {std_binom:.3f}")

In [ ]:
# Simulation: draw 1000 samples from Binomial(10, 0.5)
# As the sample size grows, the histogram should match the theoretical PMF.

samples = np.random.binomial(n=10, p=0.5, size=1000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of simulated data
axes[0].hist(samples, bins=range(0, 12), density=True, color="teal",
             edgecolor="white", alpha=0.8, label="Simulated (1000 samples)")
axes[0].stem(x, pmf_vals, basefmt=" ", linefmt="r-", markerfmt="ro",
             label="Theoretical PMF")
axes[0].set_title("Simulated vs Theoretical")
axes[0].set_xlabel("Number of Successes")
axes[0].set_ylabel("Probability")
axes[0].legend()

# CDF
cdf_vals = binom.cdf(x, n, p)
axes[1].step(x, cdf_vals, where="post", color="navy")
axes[1].set_title("Cumulative Distribution Function (CDF)")
axes[1].set_xlabel("$k$")
axes[1].set_ylabel("$P(X \\leq k)$")

plt.suptitle(f"Binomial Distribution: $n={n},\\, p={p}$", fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Normal Distribution

The **normal distribution** $X \sim N(\mu, \sigma^2)$ is the most important continuous distribution in statistics.

### Key properties

- Symmetric, bell-shaped curve centred at $\mu$
- Spread controlled by $\sigma$ — larger $\sigma$ → flatter, wider curve
- The **68–95–99.7 rule**: approximately 68%, 95%, and 99.7% of values fall within $1\sigma$, $2\sigma$, and $3\sigma$ of the mean
- The **standard normal** $Z \sim N(0, 1)$ is obtained via the **z-score**: $z = \dfrac{x - \mu}{\sigma}$

### Why is the normal distribution so important?

Many natural measurements are approximately normal. More fundamentally, the **Central Limit Theorem** (Part 3) guarantees that sample means tend to be normally distributed — making the normal the backbone of most inferential methods.

In [ ]:
# Normal curves with different means and standard deviations
# Notice: changing μ shifts the curve; changing σ changes its width.

x = np.linspace(-10, 10, 400)

plt.figure(figsize=(7, 4))
for mu_val, sigma_val, color in [(0, 1, "steelblue"), (2, 1.5, "darkorange"), (0, 2, "green")]:
    plt.plot(x, norm.pdf(x, mu_val, sigma_val),
             label=f"$\\mu={mu_val},\\; \\sigma={sigma_val}$",
             color=color, linewidth=2)

plt.title("Normal Distributions with Different Parameters")
plt.xlabel("$x$")
plt.ylabel("Density $f(x)$")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# The 68-95-99.7 rule illustrated on N(50, 10)
mu_val, sigma_val = 50, 10
x = np.linspace(mu_val - 4*sigma_val, mu_val + 4*sigma_val, 400)
y = norm.pdf(x, mu_val, sigma_val)

plt.figure(figsize=(8, 4))
plt.plot(x, y, color="black", linewidth=2)

for k, color, label in [(3, "#d4ecd4", "99.7%"), (2, "#a8d5a2", "95%"), (1, "#5aaa5a", "68%")]:
    plt.fill_between(x, y,
                     where=(x >= mu_val - k*sigma_val) & (x <= mu_val + k*sigma_val),
                     color=color, label=f"$\\mu \\pm {k}\\sigma$ ({label})")

plt.title(f"68–95–99.7 Rule: $N({mu_val}, {sigma_val}^2)$")
plt.xlabel("$x$")
plt.ylabel("Density")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Computing probabilities with the normal distribution
# Suppose heights are N(μ=170 cm, σ=8 cm).

mu_h, sigma_h = 170, 8

# P(height < 180)
p_below_180 = norm.cdf(180, mu_h, sigma_h)

# P(160 < height < 185)
p_between = norm.cdf(185, mu_h, sigma_h) - norm.cdf(160, mu_h, sigma_h)

# z-score for 185 cm
z_185 = (185 - mu_h) / sigma_h

print(f"Heights ~ N(μ={mu_h}, σ={sigma_h})")
print(f"  P(height < 180)          = {p_below_180:.4f}")
print(f"  P(160 < height < 185)    = {p_between:.4f}")
print(f"  z-score for 185 cm       = {z_185:.2f}")

In [ ]:
# Simulating normal data and checking the fit
data = np.random.normal(loc=50, scale=10, size=1000)

plt.figure(figsize=(6, 4))
sns.histplot(data, bins=30, kde=True, stat="density", color="purple", alpha=0.6)

# Overlay the theoretical density
x_range = np.linspace(data.min(), data.max(), 300)
plt.plot(x_range, norm.pdf(x_range, 50, 10), color="black",
         linewidth=2, linestyle="--", label="Theoretical $N(50, 10^2)$")

plt.title("Simulated Normal Data vs Theoretical Density")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

---
## Summary

- **Minitab vs Python**: Python libraries (`scipy.stats`, `numpy`, `pandas`) replicate every Minitab calculation with full code transparency.
- **Probability distributions** describe how probability is spread across outcomes — PMF for discrete, PDF for continuous.
- **Binomial distribution** $\text{Bin}(n, p)$: counts successes in $n$ independent trials; mean $\mu = np$, std dev $\sigma = \sqrt{np(1-p)}$.
- **Normal distribution** $N(\mu, \sigma^2)$: symmetric bell curve; the 68–95–99.7 rule; standardise with $z = (x - \mu)/\sigma$.

**Next:** Part 3 – Sampling Distributions and Statistical Inference.